# Signal Preprocessing EDA

This notebook contains the EDA used in the report: label distribution, PSD bandpower evidence, smoothing, notch filtering, high-pass baseline drift, and waveform before/after visualization.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from config import FIGURE_DIR, TABLE_DIR, REPORT_CLASSES
from signal_processing import (
    attach_multilabel_targets,
    bandpower,
    compute_psd,
    highpass_filter,
    load_metadata,
    load_wfdb_record,
    lowpass_filter,
    notch_filter,
    preprocess_record,
    smooth_signal,
)

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

MAX_RECORDS = 0
EXAMPLE_ECG_ID = 189
LEAD_INDEX = 1

## Label Distribution

In [ ]:
records, labels = attach_multilabel_targets(load_metadata())
if MAX_RECORDS > 0:
    records = records.head(MAX_RECORDS).reset_index(drop=True)
    labels = labels[:MAX_RECORDS]

label_counts = pd.DataFrame({
    "label": REPORT_CLASSES,
    "number_of_samples": labels.sum(axis=0).astype(int),
}).sort_values("number_of_samples", ascending=False)
label_counts.to_csv(TABLE_DIR / "label_distribution.csv", index=False)
label_counts

## Dataset-Scale PSD Evidence

In [ ]:
metric_rows = []
example_signal = None
example_fs = 100

for row in tqdm(records.itertuples(index=False), total=len(records), desc="Computing PSD evidence", unit="record"):
    try:
        signal, fs = load_wfdb_record(str(row.filename_lr))
        smoothed = smooth_signal(signal)
        notched = notch_filter(smoothed, fs=fs, freq=50.0)
        processed = preprocess_record(signal, fs=fs)
        metric_rows.append({
            "ecg_id": int(row.ecg_id),
            "powerline_before": bandpower(signal[:, LEAD_INDEX], fs, 49.0, min(51.0, fs / 2)),
            "powerline_after": bandpower(notched[:, LEAD_INDEX], fs, 49.0, min(51.0, fs / 2)),
            "baseline_before": bandpower(notched[:, LEAD_INDEX], fs, 0.0, 0.5),
            "baseline_after": bandpower(processed[:, LEAD_INDEX], fs, 0.0, 0.5),
        })
        if int(row.ecg_id) == EXAMPLE_ECG_ID:
            example_signal = signal
            example_fs = fs
    except Exception as exc:
        print(f"Skip ecg_id={row.ecg_id}: {exc}")

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(TABLE_DIR / "preprocessing_psd_metrics.csv", index=False)
metrics.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].boxplot([metrics["powerline_before"], metrics["powerline_after"]], tick_labels=["Before notch", "After notch"], showfliers=False)
axes[0].set_ylabel("Bandpower 49-51 Hz")
axes[0].grid(alpha=0.25)

axes[1].boxplot([metrics["baseline_before"], metrics["baseline_after"]], tick_labels=["Before high-pass", "After high-pass"], showfliers=False)
axes[1].set_ylabel("Bandpower 0-0.5 Hz")
axes[1].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "preprocessing_bandpower_boxplots.png", dpi=160, bbox_inches="tight")
plt.show()

## Representative Record: ECG 189, Lead II

In [ ]:
if example_signal is None:
    selected = records.loc[records["ecg_id"] == EXAMPLE_ECG_ID].iloc[0]
    example_signal, example_fs = load_wfdb_record(str(selected.filename_lr))

before = example_signal[:, LEAD_INDEX]
smooth = smooth_signal(before)
smooth_notch = notch_filter(smooth, fs=example_fs, freq=50.0)
after = highpass_filter(smooth_notch, fs=example_fs, cutoff=0.5)
time_axis = np.arange(before.size) / example_fs

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(time_axis, before, linewidth=1.0)
axes[0].set_title("Waveform before preprocessing")
axes[0].grid(alpha=0.3)
axes[1].plot(time_axis, after, linewidth=1.0, color="tab:orange")
axes[1].set_title("Waveform after preprocessing")
axes[1].set_xlabel("Time (s)")
axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"ecg_{EXAMPLE_ECG_ID}_waveform_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(time_axis, before, linewidth=1.0)
axes[0].set_title("Raw signal")
axes[0].grid(alpha=0.3)
axes[1].plot(time_axis, smooth, linewidth=1.0, color="tab:orange")
axes[1].set_title("After moving-average smoothing")
axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"ecg_{EXAMPLE_ECG_ID}_smoothing_step.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
f_before, p_before = compute_psd(before, example_fs)
f_after, p_after = compute_psd(smooth_notch, example_fs)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(f_before, np.maximum(p_before, 1e-12), label="Before notch")
ax.semilogy(f_after, np.maximum(p_after, 1e-12), label="After notch")
ax.axvline(50.0, linestyle="--", color="tab:red", label="50 Hz")
ax.set_xlim(45.0, min(55.0, example_fs / 2))
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"ecg_{EXAMPLE_ECG_ID}_notch_psd.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
baseline_before = lowpass_filter(smooth_notch, fs=example_fs, cutoff=0.5)
baseline_after = lowpass_filter(after, fs=example_fs, cutoff=0.5)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(time_axis, smooth_notch, linewidth=0.9, alpha=0.7, label="Before high-pass")
axes[0].plot(time_axis, baseline_before, linewidth=2.0, label="Baseline")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].plot(time_axis, after, linewidth=0.9, alpha=0.8, label="After high-pass")
axes[1].plot(time_axis, baseline_after, linewidth=2.0, label="Residual baseline")
axes[1].set_xlabel("Time (s)")
axes[1].grid(alpha=0.3)
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"ecg_{EXAMPLE_ECG_ID}_highpass_baseline.png", dpi=150, bbox_inches="tight")
plt.show()